# CS4100 Final Project  
Team Members: Khushi Khan, Dustin Zhang, Kayla Handley, Koena Gupta

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import torch
import torchvision
import torchmetrics
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split    
from torch import nn
from torch import optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [ ]:
mp3_df = pd.read_csv('../data/cleaned/fma_cleaned_dataset_emotion_labels.csv', low_memory=False)

mp3_df.head()

In [ ]:
track_ids_df = np.load('../data/cleaned/fma_track_ids.npy')
spectrograms_df = np.load('../data/cleaned/fma_spectrograms.npy')
# labels_df = np.load('../data/cleaned/fma_labels.npy') # dont know if we need this

# Inspecting the data
for i in range(3):
    track_id = track_ids_df[i]
    print(f"Track id: {track_id}")
    # print(f"Labels: {labels_df[i]}")
    print(f"Valence: {mp3_df.loc[mp3_df['track_id'] == track_id, 'valence'].item()}")
    print(f"Energy: {mp3_df.loc[mp3_df['track_id'] == track_id, 'energy'].item()}")
    plt.imshow(spectrograms_df[i], cmap='gray')
    plt.show()

In [ ]:
# Targets to predict
y = mp3_df[['valence', 'energy']].to_numpy()
n_outputs = y.shape[1]

# Passing in spectrograms
X = spectrograms_df
n_samples, n_mels, n_timeframes = X.shape
X = X.reshape(n_samples, 1, n_mels, n_timeframes)
n_samples, num_channels, n_mels, n_timeframes = X.shape

shape_info = {
    "Number of samples": n_samples,
    "Number of channels": num_channels,
    "Number of Mel frequency bands": n_mels,
    "Number of time frames": n_timeframes
}

for k, v in shape_info.items():
    print(f"{k}: {v}")

In [ ]:
# Defining a custom dataset class
class SpectrogramDataset(Dataset):
    def __init__(self, spectrograms, labels, transform=None, resize=None):
        self.spectrograms = spectrograms
        self.labels = labels
        self.transform = transform
        self.resize = resize

    def __len__(self):
        return len(self.spectrograms)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img = self.spectrograms[idx]
        valence_energy_labels = self.labels[idx]
        
        # Convert numpy array/image to Pytorch tensor
        if self.transform:
            img = self.transform(img)

        # Convert targets to Pytorch tensor type
        valence_energy_labels = torch.tensor(valence_energy_labels, dtype=torch.float32)

        if not torch.is_tensor(img) or not torch.is_tensor(valence_energy_labels):
            raise TypeError("Expected a torch.Tensor: Either the spectrogram or labels are incorrect types")
        
        if img.shape[1] != 1:
            n_timeframes, n_channels, n_mels = img.shape
            img = img.reshape(n_channels, n_timeframes, n_mels)

        # Returning (spectrogram, label_vector of valence and energy)
        return img, valence_energy_labels

In [ ]:
# Splitting data into train and test sets manually to preserve insertion order
X_split_idx = int(X.shape[0] * 0.7)
y_split_idx = int(y.shape[0] * 0.7)
X_train, X_test = X[:X_split_idx, :], X[X_split_idx:, :] # 70% training, 30% testing
y_train, y_test = y[:y_split_idx, :], y[y_split_idx:, :] # 70% training, 30% testing

emotion_classes = ('emotion_joy_excitement', 'emotion_peaceful_content', 'emotion_anger_tension', 'emotion_sadness')

In [ ]:
# Initialize the dataset
img_dims = (256, 256)
transform = transforms.ToTensor()
train_ds = SpectrogramDataset(spectrograms=X_train, 
                              labels=y_train, 
                              transform=transform, 
                              resize=img_dims)
test_ds = SpectrogramDataset(spectrograms=X_test, 
                             labels=y_test,
                             transform=transform, 
                             resize=img_dims)

# Initialize the data loaders; insertion order matters here, so we won't shuffle
train_dataloader = DataLoader(train_ds, batch_size=2, shuffle=False)
test_dataloader = DataLoader(test_ds, batch_size=2, shuffle=False)

In [ ]:
img, labels = train_ds[0]
print(img.shape)

In [ ]:
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(CNN, self).__init__()

        # 1st convolutional layer
        self.conv1 = nn.Conv2d(
            in_channels=in_channels, 
            out_channels=6,
            kernel_size=3,
            padding=1)
        
        # Max pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 2nd convolutional layer
        self.conv2 = nn.Conv2d(
            in_channels=6, 
            out_channels=16, 
            kernel_size=3,
            padding=1)

        # Fully connected layer
        self.fc1 = None

    def forward(self, x):
        x = F.relu(self.conv1(x))  # Apply first convolution and ReLU activation
        x = self.pool(x)           # Apply max pooling
        x = F.relu(self.conv2(x))  # Apply second convolution and ReLU activation
        x = self.pool(x)           # Apply max pooling
        x = x.view(x.size(0), -1)  # Flatten the tensor

        if self.fc1 is None:
            # Apply fully connected layer
            self.fc1 = nn.Linear(x.shape[1], 4).to(x.device)
        x = self.fc1(x)
        return x

device = "cuda" if torch.cuda.is_available() else "cpu"

# Since the spectrograms are grayscale, in-channels=1
model = CNN(in_channels=1, num_classes=len(emotion_classes)).to(device)
    
print(f"Model architecture:\n {model}")

In [ ]:
# Define the loss function for classification
criterion = nn.CrossEntropyLoss()

# Define the optimizer and learning rate
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
batch = next(iter(train_dataloader))
print(type(batch))
print(f"{type(batch[0])}, {type(batch[1])}")
print(f"{type(batch[0][0])}, {type(batch[0][1])}, {type(batch[1][0])}, {type(batch[1][1])}")

In [ ]:
num_epochs=10
for epoch in range(num_epochs):
 # Iterate over training batches
   print(f"Epoch [{epoch + 1}/{num_epochs}]")

   for batch_idx, batch in enumerate(tqdm(train_dataloader)):
      # Move data and targets to GPU, faster performance
      data, targets = batch
      print(type(data))
      print(type(targets))

      data = data.to(device)
      targets = targets.to(device)

      print(data.shape)

      print(data)

      # Predicted output
      scores = model(data)

      # Calculating Cross Entropy Loss
      loss = criterion(scores, targets)

      # Accumulate gradients
      optimizer.zero_grad()

      # Computes the gradients of the loss w.r.t. model parameters/Backward pass
      loss.backward()

      # Update the weights
      optimizer.step()